# Methoden en Technieken -- Blok 2

## Datapunt 2a,

In deze opdracht worden de volgende leeruitkomsten getoetst, relevante termen zijn **dik** gedrukt:
- A2: Je stelt voor een AI-oplossing juridische, ethische, organisatorische, **functionele en
technische requirements** op.
- B1: Je **verkent en prepareert een dataset voor het trainen en testen van een AI-model en
kan de voor- en nadelen van het gebruik van een bestaande dataset onderbouwen**, rekening
houdend met technische en ethische randvoorwaarden.
- B2: Je **stelt op basis van requirements en data een geschikte architectuur voor een AI-
oplossing op en selecteert daarvoor passende AI-technieken gebruik makend van bijvoorbeeld**
machine learning, **deep learning**, kennisrepresentatie, computer vision en **natural language
processing**.
- B3: Je **ontwikkelt een nieuw** of voorgetraind **AI-model volgens een iteratief en systematisch
proces**.
- C2: **Je evalueert en beoordeelt de kwaliteit van een AI-model aan de hand van kwaliteitscriteria die in het vakgebied erkend worden** zoals robustness, **performance**, scalability,
explainability, **model complexity** en resource demand.

<hr>

- Dit is een individuele opdracht. Het is toegestaan met elkaar de opdracht te bespreken, maar iedereen schrijft zijn/haar eigen code en antwoorden. 
- Mocht je code van het internet hebben overgenomen/aangepast, vermeld dan de bron in het commentaar.
- Mocht je naar een bron verwijzen voor jouw argumentatie, vermeld dan de bron *en* copy-paste de specifieke uitspraak uit de bron waar je het argument op baseert. 
- Het gebruik van generatieve AI (Chat-GPT, etc...) is *alleen* toegestaan bij de voorbereiding op de opdracht. Denk hierbij aan activiteiten zoals brainstormen, of het maken van een plan. Het is niet toegestaan code of tekst van generative AI over te nemen. (Level 2 van de AI Assessment Scale)



## De opdracht

In de slides van les 3 staat hoe een Engels - Nederlands vertaalmodel gemaakt wordt met een transformer-encoder en -decoder. Zie ook hoofdstuk 15 uit Deep Learning with Python (derde editie).

### Deel 1
#### Deel 1a
Maak dit model na en fit dit model.

### Deel 1a — Reproduceerbare baseline (les 3)
Deze uitwerking reconstrueert de **Transformer encoder-decoder** uit `Blok2_Les03.ipynb` (modelklassen uit les 3). De dataset bestaat uit Engels–Nederlandse Anki/ManyThings-zinsparen. We gebruiken de originele 15.000 tokens, 20 tokenposities, 256 embeddingdimensies, 2056 feedforwarddimensies, 8 attention-heads en batchgrootte 64. De uitvoer wordt pas zichtbaar **na het uitvoeren van de code**; er zijn nog geen trainingsresultaten geproduceerd.

**Uitvoering:** voer de cellen van boven naar beneden uit. De eerste datasetdownload vereist internet. Installeer indien nodig `tensorflow keras numpy matplotlib pandas`. Train- en validatieresultaten worden in deel 1a getoond; de testset blijft onaangeroerd tot deel 3.

In [ ]:
import random
import string
import re
import json
import zipfile
from pathlib import Path

import keras
from keras import layers, ops
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 1407
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

DATA_DIR = Path("data/raw")
MODEL_DIR = Path("models")
RESULTS_DIR = Path("results")
FIGURES_DIR = RESULTS_DIR / "figures"
for folder in (DATA_DIR, MODEL_DIR, RESULTS_DIR, FIGURES_DIR):
    folder.mkdir(parents=True, exist_ok=True)

print("TensorFlow:", tf.__version__, "| Keras:", keras.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

#### 1. Data laden en splitsen
De bron is de `nld-eng.zip`-dataset van ManyThings/Anki (zoals in les 3). **Download deze handmatig**, pak `nld.txt` uit en zet het bestand in `data/raw/nld.txt`. Het notebook probeert niet meer automatisch te downloaden (de website kan HTTP 406 retourneren). De eerste twee tabgescheiden kolommen zijn Engelse en Nederlandse zinnen; een eventuele derde kolom bevat broninformatie. Nederlandse doelzinnen krijgen `[start]` en `[end]`-tokens.


In [ ]:
dataset_path = DATA_DIR / "nld.txt"

# Download de oorspronkelijke Engels-Nederlandse dataset handmatig via:
# https://www.manythings.org/anki/nld-eng.zip
# Pak nld.txt uit en plaats het bestand in data/raw/nld.txt.
# De automatische download is verwijderd omdat de server HTTP 406 kan retourneren.
if not dataset_path.is_file():
    raise FileNotFoundError(
        f"Dataset niet gevonden: {dataset_path.resolve()}\n"
        "Download nld-eng.zip via https://www.manythings.org/anki/nld-eng.zip, "
        "pak nld.txt uit en plaats het in data/raw/."
    )

text_pairs = []
with open(dataset_path, encoding="utf-8") as f:
    for line in f:
        fields = line.rstrip("\r\n").split("\t")
        if len(fields) >= 2 and fields[0] and fields[1]:
            text_pairs.append((fields[0], f"[start] {fields[1]} [end]"))

assert text_pairs, "Geen zinsparen ingelezen: controleer het datasetbestand."
random.Random(SEED).shuffle(text_pairs)
val_samples = int(0.15 * len(text_pairs))
train_samples = len(text_pairs) - 2 * val_samples
train_pairs = text_pairs[:train_samples]
val_pairs = text_pairs[train_samples:train_samples + val_samples]
test_pairs = text_pairs[train_samples + val_samples:]
print("Totaal:", len(text_pairs), "| train:", len(train_pairs), "| validatie:", len(val_pairs), "| test:", len(test_pairs))
print("Voorbeeld:", train_pairs[0])

#### 2. Datasetvisualisatie
Deze grafieken tonen de omvang en zinslengtes **vóór training**, berekend uit de ingelezen dataset; het zijn geen gesimuleerde waarden. Ze ondersteunen de onderbouwing van de maximumlengte van 20 tokens.

In [ ]:
split_counts = {"Train": len(train_pairs), "Validatie": len(val_pairs), "Test (gereserveerd)": len(test_pairs)}
fig, ax = plt.subplots(figsize=(8, 3.5))
bars = ax.bar(split_counts.keys(), split_counts.values(), width=0.5)
ax.bar_label(bars, padding=3)
ax.set_title("Datasetverdeling — Engels–Nederlands")
ax.set_ylabel("Aantal zinsparen")
ax.grid(axis="y", alpha=0.2)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "01_datasetverdeling.png", dpi=160)
plt.show()

english_lengths = np.array([len(s.split()) for s, _ in train_pairs])
dutch_lengths = np.array([len(s.split()) for _, s in train_pairs])
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(english_lengths, bins=range(1, 51), alpha=0.6, label="Engels")
ax.hist(dutch_lengths, bins=range(1, 51), alpha=0.5, label="Nederlands incl. start/eind")
ax.axvline(20, linestyle="--", color="black", label="Max. invoerlengte = 20")
ax.set_xlim(0, 50)
ax.set_xlabel("Woorden per zin (ruwe spatiesplitsing)")
ax.set_ylabel("Frequentie")
ax.set_title("Verdeling van zinslengtes — trainingsset")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / "02_zinslengtes.png", dpi=160)
plt.show()

#### 3. TextVectorization en teacher forcing
De les gebruikt een Engelse `TextVectorization` met 20 posities en een Nederlandse met 21 posities. Daardoor kan de decoder invoertokens `[:-1]` gebruiken en wordt `[1:]` het target (één positie verschoven). Padding krijgt een samplegewicht van nul.

In [ ]:
strip_chars = string.punctuation + "¿"
strip_chars = strip_chars.replace("[", "")
strip_chars = strip_chars.replace("]", "")

def custom_standardization(input_string):
    lowercase = tf.strings.lower(input_string)
    return tf.strings.regex_replace(
        lowercase, f"[{re.escape(strip_chars)}]", ""
    )

vocab_size = 15000
sequence_length = 20

english_tokenizer = layers.TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length,
)
dutch_tokenizer = layers.TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length + 1,
    standardize=custom_standardization,
)
train_english_texts = [pair[0] for pair in train_pairs]
train_dutch_texts = [pair[1] for pair in train_pairs]
english_tokenizer.adapt(train_english_texts)
dutch_tokenizer.adapt(train_dutch_texts)

In [ ]:
batch_size = 64

def format_dataset(eng, spa):
    eng = english_tokenizer(eng)
    spa = dutch_tokenizer(spa)
    features = {"english": eng, "dutch": spa[:, :-1]}
    labels = spa[:, 1:]
    sample_weights = labels != 0
    return features, labels, sample_weights

def make_dataset(pairs):
    eng_texts, spa_texts = zip(*pairs)
    eng_texts = list(eng_texts)
    spa_texts = list(spa_texts)
    dataset = tf.data.Dataset.from_tensor_slices((eng_texts, spa_texts))
    dataset = dataset.batch(batch_size)
    dataset = dataset.map(format_dataset, num_parallel_calls=4)
    return dataset.cache().shuffle(2048, seed=SEED).prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset(train_pairs)
val_ds = make_dataset(val_pairs)

for features, targets, weights in train_ds.take(1):
    print('Encoder:', features['english'].shape, 'Decoder:', features['dutch'].shape, 'Targets:', targets.shape)
    print('Paddinggewicht (eerste 10):', weights[0, :10].numpy())

#### 4. Encoder, decoder en positionele embeddings
De drie modelklassen hieronder zijn gebaseerd op de voorbeeldcode van **les 3**. Hun werking is ongewijzigd; dit is bewust de baseline voor latere modeliteraties.

In [ ]:
class TransformerEncoder(keras.Layer):
    def __init__(self, hidden_dim, intermediate_dim, num_heads, **kwargs):
        super().__init__()
        key_dim = hidden_dim // num_heads
        self.self_attention = layers.MultiHeadAttention(num_heads, key_dim)
        self.self_attention_layernorm = layers.LayerNormalization()
        self.feed_forward_1 = layers.Dense(intermediate_dim, activation="relu")
        self.feed_forward_2 = layers.Dense(hidden_dim)
        self.feed_forward_layernorm = layers.LayerNormalization()

    def call(self, source, source_mask):
        residual = x = source
        mask = source_mask[:, None, :]
        x = self.self_attention(query=x, key=x, value=x, attention_mask=mask)
        x = x + residual
        x = self.self_attention_layernorm(x)
        residual = x
        x = self.feed_forward_1(x)
        x = self.feed_forward_2(x)
        x = x + residual
        x = self.feed_forward_layernorm(x)
        return x

In [ ]:
class TransformerDecoder(keras.Layer):
    def __init__(self, hidden_dim, intermediate_dim, num_heads, **kwargs):
        super().__init__()
        key_dim = hidden_dim // num_heads
        self.self_attention = layers.MultiHeadAttention(num_heads, key_dim)
        self.self_attention_layernorm = layers.LayerNormalization()
        self.cross_attention = layers.MultiHeadAttention(num_heads, key_dim)
        self.cross_attention_layernorm = layers.LayerNormalization()
        self.feed_forward_1 = layers.Dense(intermediate_dim, activation="relu")
        self.feed_forward_2 = layers.Dense(hidden_dim)
        self.feed_forward_layernorm = layers.LayerNormalization()

    def call(self, target, source, source_mask):
        residual = x = target
        x = self.self_attention(query=x, key=x, value=x, use_causal_mask=True)
        x = x + residual
        x = self.self_attention_layernorm(x)
        residual = x
        mask = source_mask[:, None, :]
        x = self.cross_attention(
            query=x, key=source, value=source, attention_mask=mask
        )
        x = x + residual
        x = self.cross_attention_layernorm(x)
        residual = x
        x = self.feed_forward_1(x)
        x = self.feed_forward_2(x)
        x = x + residual
        x = self.feed_forward_layernorm(x)
        return x

In [ ]:
class PositionalEmbedding(keras.Layer):
    def __init__(self, sequence_length, input_dim, output_dim, **kwargs):
        super().__init__()
        self.token_embeddings = layers.Embedding(input_dim, output_dim)
        self.position_embeddings = layers.Embedding(sequence_length, output_dim)

    def call(self, inputs):
        positions = ops.cumsum(ops.ones_like(inputs), axis=-1) - 1
        embedded_tokens = self.token_embeddings(inputs)
        embedded_positions = self.position_embeddings(positions)
        return embedded_tokens + embedded_positions

#### 5. Model samenstellen
Modelinstellingen: `hidden_dim=256`, `intermediate_dim=2056`, `num_heads=8`, dropout 0,5. De naam van de Nederlandse modelinput is verduidelijkt van `spanish` naar `dutch`.

In [ ]:
hidden_dim = 256
intermediate_dim = 2056
num_heads = 8

source = keras.Input(shape=(None,), dtype="int32", name="english")
x = PositionalEmbedding(sequence_length, vocab_size, hidden_dim)(source)
encoder_output = TransformerEncoder(hidden_dim, intermediate_dim, num_heads)(
    source=x,
    source_mask=source != 0,
)

target = keras.Input(shape=(None,), dtype="int32", name="dutch")
x = PositionalEmbedding(sequence_length, vocab_size, hidden_dim)(target)
x = TransformerDecoder(hidden_dim, intermediate_dim, num_heads)(
    target=x,
    source=encoder_output,
    source_mask=source != 0,
)
x = layers.Dropout(0.5)(x)
target_predictions = layers.Dense(vocab_size, activation="softmax")(x)
transformer = keras.Model([source, target], target_predictions, name='Transformer_EN_NL_Baseline')
transformer.summary()
print('Aantal parameters:', transformer.count_params())

#### 6. Compileren en fitten
De les gebruikt Adam, sparse categorical crossentropy en gewogen token-accuracy. De fit-code uit les 3 was uitgecommentarieerd en bevatte `epochs=2` (met commentaar `was 30`). Voor de **definitieve baseline** staat hieronder 30 epochs. Voor een snelle rooktest kun je `EPOCHS=2` zetten en daarna opnieuw trainen vanaf een vers geïnitialiseerd model; beschouw die rooktest niet als je definitieve baseline.

**Reproduceerbaarheid:** de geshuffelde datasplit en random seed blijven vast. Zorg dat de training daadwerkelijk is voltooid voordat je resultaten interpreteert.

In [ ]:
EPOCHS = 30
transformer.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    weighted_metrics=["accuracy"],
)

history = transformer.fit(
    train_ds,
    epochs=EPOCHS,
    validation_data=val_ds,
    verbose=1,
)

transformer.save(MODEL_DIR / "transformer_baseline.keras")
with open(RESULTS_DIR / "transformer_baseline_history.json", "w", encoding="utf-8") as f:
    json.dump({key: [float(x) for x in values] for key, values in history.history.items()}, f, indent=2)
print("Model en trainingsgeschiedenis opgeslagen.")

#### 7. Resultaten visualiseren: learning curves
De grafieken worden **uitsluitend** opgebouwd uit de werkelijk verkregen `history.history`-waarden. Daarbij vergelijken we training en validatie; testresultaten worden nog niet gebruikt.

In [ ]:
metrics = history.history
accuracy_key = next((k for k in metrics if k == "accuracy" or k.endswith("_accuracy") and not k.startswith("val_")), None)
val_accuracy_key = f"val_{accuracy_key}" if accuracy_key is not None else None
x = np.arange(1, len(metrics["loss"]) + 1)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(x, metrics["loss"], marker="o", markersize=2.5, label="Train loss")
axes[0].plot(x, metrics["val_loss"], marker="o", markersize=2.5, label="Validatie loss")
axes[0].set(xlabel="Epoch", ylabel="Sparse categorical crossentropy", title="Loss per epoch")
axes[0].legend()
axes[0].grid(alpha=0.2)

if accuracy_key and val_accuracy_key in metrics:
    axes[1].plot(x, metrics[accuracy_key], marker="o", markersize=2.5, label="Train token-accuracy")
    axes[1].plot(x, metrics[val_accuracy_key], marker="o", markersize=2.5, label="Validatie token-accuracy")
    axes[1].legend()
axes[1].set(xlabel="Epoch", ylabel="Gewogen token-accuracy", title="Accuracy per epoch")
axes[1].grid(alpha=0.2)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "03_learning_curves.png", dpi=160)
plt.show()

best_epoch = int(np.argmin(metrics["val_loss"])) + 1
print(f"Beste validatie-loss: {min(metrics['val_loss']):.4f} op epoch {best_epoch}")
if val_accuracy_key in metrics:
    print(f"Beste validatie-token-accuracy: {max(metrics[val_accuracy_key]):.4f}")
print("Let op: token-accuracy is geen maat voor volledige vertaalcorrectheid.")

#### 8. Kwalitatieve vertaalcontrole (validatieset)
Voorbeeldvertalingen worden gegenereerd door greedy decoding met het **getrainde** model. We gebruiken een kleine vaste steekproef uit de *validatieset*, niet de testset. Resultaten verschillen mogelijk door systeem en trainingsrun. De output is een controle, geen vervanging van een formele vertaalmetric.

In [ ]:
dutch_vocab = dutch_tokenizer.get_vocabulary()
dutch_index_lookup = dict(enumerate(dutch_vocab))

def translate_sentence(english_sentence):
    encoder_tokens = english_tokenizer([english_sentence])
    decoded = "[start]"
    for i in range(sequence_length):
        decoder_tokens = dutch_tokenizer([decoded])[:, :-1]
        probabilities = transformer.predict([encoder_tokens, decoder_tokens], verbose=0)
        token_id = int(np.argmax(probabilities[0, i, :]))
        token = dutch_index_lookup[token_id]
        if token in ("[end]", ""):
            break
        decoded += " " + token
    return decoded.replace("[start]", "", 1).strip()

sample_rng = random.Random(SEED)
examples = sample_rng.sample(val_pairs, k=min(8, len(val_pairs)))
rows = []
for english, reference in examples:
    rows.append({
        "Engels": english,
        "Referentie NL": reference.replace("[start]", "").replace("[end]", "").strip(),
        "Modelvertaling": translate_sentence(english),
    })
translations_df = pd.DataFrame(rows)
display(translations_df)
translations_df.to_csv(RESULTS_DIR / "voorbeeldvertalingen_validatie.csv", index=False)

print("Baseline gereed: controleer de learning curves en fouttypen vóór een modelverbetering.")

**Interpretatie na uitvoeren (nog in te vullen op basis van eigen resultaten)**
- Welke epoch heeft de laagste validatie-loss? Wordt de kloof met de train-loss groter?
- Welke zinnen worden goed of slecht vertaald? Komt dat door zinslengte, woordkeuze of grammatica?
- Welke verbetering is hieruit te onderbouwen voor deel 2? Verander per iteratie slechts één factor.

**Bronnen:** `Blok2_Les03.ipynb` (voorbeeldcode en instellingen), [ManyThings/Anki](https://www.manythings.org/anki/) (dataset), [Deep Learning with Python](https://deeplearningwithpython.io/) (lesverwijzing).

#### Deel 1b
Bespreek de voor- en nadelen van de gebruikte dataset.

> Jouw antwoord hier

### Deel 2
We willen dit model nu verbeteren. Bedenk een (mogelijke) verbetering. Het is niet de bedoeling dat je bestaande pretrained modellen gebruikt. Train alles zelf.

#### Deel 2a
Stel een korte lijst van eisen op waar jouw verbeterde model aan hoort te voldoen. 

> Jouw antwoord hier

#### Deel 2b
Licht het volgende toe:
- Welke verbetering je uitvoert en waarom je denkt dat dit voor een verbetering zal zorgen.
- Welke maat je gebruikt om de verbetering te meten, en waarom.

Houd je antwoorden kort en bondig.

> Jouw antwoord hier

#### Deel 2c
Train dit verbeterde model.

### Deel 3
#### Deel 3a
Evalueer het model.

#### Deel 3b
Bespreek het volgende.
- Hoeveel je model verbeterd (of verslechterd) is
- In het geval van geen verbetering, een mogelijke verklaring waarom het niet werkte.
- Licht toe wat je volgende stap zou zijn om betere resultaten te halen. Deze stap hoef je niet uit te voeren, maar wees wel specifiek en verantwoord het met jouw resultaten.

Houd je antwoorden kort en bondig.

> Jouw antwoord hier